In [1]:
import numpy as np
import matplotlib.pyplot as plt
from openalea.mtg import MTG


def load_mtg(rsml_path):
    from rsml import rsml2mtg
    return rsml2mtg(rsml_path)


def intercept_curve(mtg:MTG, plant_id=1, time=None, nlengths=2500, step=1e-3):
    """
    Calcule la courbe intercepto pour une plante d'un mtg, éventuellement à un temps donné.
    """
    from hydroroot.analysis import intercept
    from hydroroot.hydro_io import import_rsml_to_discrete_mtg
    if time is not None:
        from mtgutils import mtg_at_time_t
        sub_mtg = mtg.sub_mtg(plant_id)
        mtg_at_t = mtg_at_time_t(sub_mtg, time)
        mtg_test = import_rsml_to_discrete_mtg(mtg_at_t)
    else:
        sub_mtg = mtg.sub_mtg(plant_id)
        mtg_test = import_rsml_to_discrete_mtg(sub_mtg)
    lengths = np.linspace(0, (nlengths-1)*step, nlengths)
    intercepto = intercept(g=mtg_test, dists=lengths, dl=3e-3, max_order=None)
    return lengths, intercepto

def intercept_curve_at_all_time(mtg:MTG, plant_id=1, nlengths=2500, step=1e-3):
    """
    Calcule la courbe intercepto pour une plante d'un mtg, éventuellement à un temps donné.
    """
    from hydroroot.analysis import intercept
    from hydroroot.hydro_io import import_rsml_to_discrete_mtg
    times = mtg.properties()["time"]
    # get max time value from dict
    max_time = max(max(times.values()))
    times = [i for i in range(1, int(max_time)+1)]
    lengths = np.linspace(0, (nlengths-1)*step, nlengths)
    intercepto_all = []
    for time in times:
        from mtgutils import mtg_at_time_t
        sub_mtg = mtg.sub_mtg(plant_id)
        mtg_at_t = mtg_at_time_t(sub_mtg, time)
        mtg_test = import_rsml_to_discrete_mtg(mtg_at_t)
        intercepto = intercept(g=mtg_test, dists=lengths, dl=3e-3, max_order=None)
        intercepto_all.append(intercepto)
    intercepto_all = np.array(intercepto_all)
    return lengths, intercepto_all

def get_cmap(n, name='tab20'):
    """
    Get a colormap with n unique colors.
    """
    cmap = plt.get_cmap(name)
    if n <= cmap.N:
        colors = [cmap(i) for i in range(n)]
    else:
        # Interpolate in the colormap if n > cmap.N
        colors = [cmap(i / n) for i in range(n)]
    return colors

def plot_interceptos(curves, labels=None, title="Intercepto vs Length"):
    """
    Affiche plusieurs courbes intercepto sur le même graphique, avec palette variée.
    """
    plt.figure(figsize=(10, 6))
    n = len(curves)
    colors = get_cmap(n, name='tab20')
    for i, (lengths, intercepto) in enumerate(curves):
        label = labels[i] if labels else f"Curve {i+1}"
        plt.plot(lengths, intercepto, label=label, color=colors[i], lw=2, alpha=0.9)
    plt.xlabel("Length")
    plt.ylabel("Intercepto")
    plt.title(title)
    plt.legend(loc='upper left', bbox_to_anchor=(1,1))
    plt.tight_layout(rect=[0,0,0.85,1])
    plt.grid(alpha=0.3)
    plt.show()

def plot_interceptos_3d(curves, labels=None, title="Intercepto vs Length (3D)"):
    """
    Affiche plusieurs courbes intercepto sur le même graphique en 3D, palette colorée.
    """
    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection='3d')
    n = len(curves)
    colors = get_cmap(n, name='tab20')
    for i, (lengths, intercepto) in enumerate(curves):
        label = labels[i] if labels else f"Curve {i+1}"
        ax.plot(lengths, [i+1]*len(lengths), intercepto, label=label,
                color=colors[i], lw=2, alpha=0.85)
    ax.set_xlabel("Length")
    ax.set_ylabel("Time (index)")
    ax.set_zlabel("Intercepto")
    plt.title(title)
    ax.legend(loc='upper left', bbox_to_anchor=(1,1))
    plt.tight_layout(rect=[0,0,0.85,1])
    plt.show()

def plot_interceptos_slider(lengths, interceptos, title="Intercepto vs Length (Slider)"):
    """
    Affiche les interceptos dans un slider (nécessite ipywidgets).
    Fonctionne uniquement dans un environnement Jupyter Notebook.
    """
    try:
        import matplotlib.pyplot as plt
        from ipywidgets import interact, IntSlider
        from IPython.display import display
    except ImportError:
        print("ipywidgets and IPython are required for the slider. Please install them.")
        return

    n = interceptos.shape[0]

    def plot_idx(idx):
        plt.figure(figsize=(8,5))
        plt.plot(lengths, interceptos[idx], lw=2, color="tab:blue")
        plt.xlabel("Length")
        plt.ylabel("Intercepto")
        plt.title(f"{title} | t={idx+1}")
        plt.grid(alpha=0.3)
        plt.show()

    slider = IntSlider(min=0, max=n-1, step=1, value=0, description='Time index')
    interact(plot_idx, idx=slider)
    display(slider)


In [2]:
rsml_1 = "/home/loai/Images/DataTest/UC1_data/230629PN026/61_graph.rsml"
rsml_2 = None  # Facultatif
time_1 = 15
time_2 = None
plant_id_1 = 1
plant_id_2 = 1

# Charger et calculer
curves = []
labels = []

mtg1 = load_mtg(rsml_1)
print("plant_ids", mtg1.vertices(scale=1))
l1, i1 = intercept_curve(mtg1, plant_id=plant_id_1, time=time_1)
#curves.append((l1, i1))
#labels.append(f"RSML 1{' (t='+str(time_1)+')' if time_1 else ''}")

l1_time, i1_time = intercept_curve_at_all_time(mtg1, plant_id=plant_id_1)
for i, intercepto in enumerate(i1_time):
    curves.append((l1_time, intercepto))
    labels.append(f"t={i+1}")

if rsml_2:
    mtg2 = load_mtg(rsml_2)
    l2, i2 = intercept_curve(mtg2, plant_id=plant_id_2, time=time_2)
    curves.append((l2, i2))
    labels.append(f"RSML 2{' (t='+str(time_2)+')' if time_2 else ''}")

plant_ids [1, 30, 43, 64, 88]


Computing intercepts: 100%|██████████| 2500/2500 [00:01<00:00, 1996.12it/s]


In [3]:
plot_interceptos_slider(l1_time, i1_time, title="Courbes intercepto (Slider)")

interactive(children=(IntSlider(value=0, description='Time index', max=28), Output()), _dom_classes=('widget-i…

IntSlider(value=0, description='Time index', max=28)

In [4]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def subplot_intercepto_image(lengths, interceptos, img_stack, plant_id=1, title="Intercepto"):
    T = min(len(interceptos), len(img_stack))
    def show_idx(idx):
        plt.figure(figsize=(12,5))
        # Courbe intercepto
        plt.subplot(1,2,1)
        plt.plot(lengths, interceptos[idx], lw=2, color="tab:blue")
        plt.xlabel("Length")
        plt.ylabel("Intercepto")
        plt.title(f"Intercepto of plant {plant_id} | t={idx+1}")
        plt.grid(alpha=0.3)
        # Image racine
        plt.subplot(1,2,2)
        plt.imshow(img_stack[idx], cmap='gray')
        plt.axis('off')
        plt.title(f"Image stack t={idx+1}")
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()
    interact(show_idx, idx=IntSlider(min=0, max=T-1, step=1, value=0, description="Time index"))


In [ ]:
import tifffile
img_stack_path = "/home/loai/Images/DataTest/UC1_data/230629PN026/22_registered_stack.tif"
img_stack = tifffile.imread(img_stack_path)  # shape: (n_time, H, W)
subplot_intercepto_image(l1_time, i1_time, img_stack, title="Visu Intercepto et Image Racine", plant_id=plant_id_1)


interactive(children=(IntSlider(value=0, description='Time index', max=28), Output()), _dom_classes=('widget-i…